In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

RS = 0.001
RCT = 0.001
CDL = 500

tau = RCT*CDL
print(f"Time constant is {tau} seconds\n")

In [ ]:
def time_domain_model(RS, RCT, CDL, I_signal, t):
    
    dt = t[1] - t[0]
    I_1 = np.zeros(len(t))
    V = np.zeros(len(t))
    alpha = np.exp(-dt/(RCT * CDL))
    
    
    for k in range(1,len(t)):
        I_1[k] = alpha * I_1[k - 1] + (1 - alpha) * I_signal[k]

        
        V[k] = I_1[k] * RCT + I_signal[k] * RS
    

    return V, I_1


In [ ]:
freqs = np.logspace(-1, 3, 10) 

Z_real_sim = []
Z_imag_sim = []

for f in freqs:
    t_sweep = np.linspace(0, 10/f, 10000)
    omega = 2 * np.pi * f
    I_sweep = np.sin(omega * t_sweep)
    
    V_sweep, _ = time_domain_model(RS, RCT, CDL, I_sweep, t_sweep)
    
    N = len(t_sweep)
    V_fft = np.fft.fft(V_sweep)
    I_fft = np.fft.fft(I_sweep)
    
    dt = t_sweep[1] - t_sweep[0]
    freqs_fft = np.fft.fftfreq(N, dt)
    idx = np.argmin(np.abs(freqs_fft - f))
    
    Z = V_fft[idx] / I_fft[idx]
    Z_real_sim.append(Z.real)
    Z_imag_sim.append(-Z.imag)


plt.figure(figsize=(8, 6))
plt.plot(Z_real_sim, Z_imag_sim, 'bo-', markersize=5)
plt.xlabel('Z real (Ohms)')
plt.ylabel('-Z imaginary (Ohms)')
plt.title('Nyquist Plot from Time Domain Simulation')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
def freq_domain_model(freqs, RS, RCT, CDL):
    omega = 2 * np.pi * freqs
    Z_cdl = 1 / (1j * omega * CDL)
    Z_parallel = (RCT * Z_cdl) / (RCT + Z_cdl)
    return RS + Z_parallel

# Calculate frequency domain model
Z_freq = freq_domain_model(freqs, RS, RCT, CDL)

# Plot both on same graph
plt.figure(figsize=(8, 6))
plt.plot(Z_real_sim, Z_imag_sim, 'bo-', markersize=5, label='Time Domain Simulation')
plt.plot(Z_freq.real, -Z_freq.imag, 'r--', linewidth=2, label='Frequency Domain Model')
plt.xlabel('Z real (Ohms)')
plt.ylabel('-Z imaginary (Ohms)')
plt.title('Nyquist Plot - Time Domain vs Frequency Domain')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

In [ ]:
Z_freq = freq_domain_model(freqs, RS, RCT, CDL)
Z_sim = np.array(Z_real_sim) - 1j * np.array(Z_imag_sim)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

ax1.semilogx(freqs, 20 * np.log10(np.abs(Z_freq)), 'r--', linewidth=2, label='Frequency Domain')
ax1.semilogx(freqs, 20 * np.log10(np.abs(Z_sim)), 'bo-', markersize=4, label='Time Domain')
ax1.set_xlabel('Frequency (Hz)')
ax1.set_ylabel('Magnitude (dB)')
ax1.set_title('Bode Plot - Magnitude')
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.legend()

ax2.semilogx(freqs, np.angle(Z_freq, deg=True), 'r--', linewidth=2, label='Frequency Domain')
ax2.semilogx(freqs, np.angle(Z_sim, deg=True), 'bo-', markersize=4, label='Time Domain')
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Phase (degrees)')
ax2.set_title('Bode Plot - Phase')
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()

plt.tight_layout()
plt.show()